In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("../utils"))
sys.path.insert(0, os.path.abspath("../sdlarch-rl"))


from sdlarch_rl.utils.utils import get_last_index, RealExcludeButtonsWrapper, GenericCNN, TimeLimit, FrameSkip
from sdlarch_rl import make
from stable_baselines3.common.atari_wrappers import WarpFrame
import time
import cv2
import numpy as np
import keyboard
from stable_baselines3.common.atari_wrappers import WarpFrame
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from pathlib import Path
import pygame
from inputs import get_gamepad, devices
import threading
from imitation.data.types import Trajectory
import torch as th
import os
import re
from utils import get_last_index, LSTMWrapper
import keyboard
from mario import make_env


env = make_vec_env(make_env(human=True, width=800, height=600), n_envs=1, vec_env_cls=DummyVecEnv)

gamepad = devices.gamepads[0]
STATE = {k: 0 for k in ["UP", "DOWN", "LEFT", "RIGHT", "A", "B", "X", "Y", "START", "CAM_X", "CAM_Y", "LT", "RT", "L3"]}
lock = threading.Lock()

DEADZONE = 10000
NORM = 32768

def input_loop():
    while True:
        for e in gamepad.read():
            with lock:
                if e.code == "ABS_HAT0X":
                    STATE["LEFT"]  = 1 if e.state == -1 else 0
                    STATE["RIGHT"] = 1 if e.state == 1 else 0

                elif e.code == "ABS_HAT0Y":
                    STATE["UP"]   = 1 if e.state == -1 else 0
                    STATE["DOWN"] = 1 if e.state == 1 else 0

                elif e.code == "ABS_X":
                    if e.state > DEADZONE:
                        STATE["LEFT"] = 0
                        STATE["RIGHT"] = 1
                    elif e.state < -DEADZONE:
                        STATE["LEFT"] = 1
                        STATE["RIGHT"] = 0
                    else:
                        STATE["LEFT"] = 0
                        STATE["RIGHT"] = 0

                elif e.code == "ABS_Y":
                    if e.state > DEADZONE:
                        STATE["UP"] = 1
                        STATE["DOWN"] = 0
                    elif e.state < -DEADZONE:
                        STATE["UP"] = 0
                        STATE["DOWN"] = 1
                    else:
                        STATE["UP"] = 0
                        STATE["DOWN"] = 0

                # RIGHT STICK X
                elif e.code == "ABS_RX":
                    if abs(e.state) > DEADZONE:
                        STATE["CAM_X"] = e.state / NORM
                    else:
                        STATE["CAM_X"] = 0

                # RIGHT STICK Y
                elif e.code == "ABS_RY":
                    if abs(e.state) > DEADZONE:
                        STATE["CAM_Y"] = e.state / NORM
                    else:
                        STATE["CAM_Y"] = 0

                elif e.code == "BTN_SOUTH":
                    STATE["A"] = e.state

                elif e.code == "BTN_NORTH":
                    STATE["Y"] = e.state

                elif e.code == "BTN_EAST":
                    STATE["B"] = e.state

                elif e.code == "BTN_WEST":
                    STATE["X"] = e.state

                elif e.code == "BTN_START" or e.code == "BTN_MENU" \
                    or e.code == "BTN_SELECT" or e.code == "BTN_MODE"  \
                    or e.code == "BTN_OPTIONS":
                    STATE["START"] = e.state

                elif e.code == "BTN_THUMBL":
                    STATE["L3"] = e.state

                elif e.code == "ABS_Z":
                    value = e.state / 255.0
                    if value < 0.5:
                        value = 0
                    STATE["LT"] = value
                
                elif e.code == "ABS_RZ":
                    value = e.state / 255.0
                    if value < 0.5:
                        value = 0
                    STATE["RT"] = value

t_input = threading.Thread(target=input_loop, daemon=True)
t_input.start()

clock = pygame.time.Clock()

env.reset()
while True:
    action = [np.zeros(4, dtype=np.float32)]

    with lock:
        received_action = [
            STATE["UP"], # 0 
            STATE["DOWN"], 
            STATE["LEFT"], 
            STATE["RIGHT"],
            STATE["A"], 
            STATE["B"], 
            STATE["X"],
            STATE["Y"], # 7
            STATE["LT"],
            STATE["RT"],
            STATE["L3"],
            STATE["CAM_X"], # 10
            STATE["CAM_Y"], # 11
            STATE["START"],
        ]


        if keyboard.is_pressed('left') or received_action[2]: action[0][0] = 1
        if keyboard.is_pressed('right') or received_action[3]: action[0][1] = 1
        if received_action[6]: action[0][2] = 1 #run
        if received_action[4]: action[0][3] = 1 #jump

    if keyboard.is_pressed('s'):
        state = env.envs[0].unwrapped.em.get_state()
        with open("default.state", "wb") as f:
            f.write(state)
        break


    observation, reward, terminated, info = env.step(action)
    
    clock.tick(60)

env.close()

D:\anaconda3\envs\env311\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


Pygame initialized: 800x600
statename is None setting to default state
